# A deep language cortex: what does brain-plausible learning cost?

Built from commit `9d54be1`. One network (8-word context, 3 hidden layers of 512,
16,000-word vocabulary), three
learning rules - backprop, predictive coding, predictive coding with its own learned feedback pathway
(Kolen-Pollack) - against a Kneser-Ney 5-gram, on WikiText-103. See `PREREGISTRATION_deepcortex.md`.

In [ ]:
import os, sys, time, glob, subprocess
W = "/kaggle/working/brain"
for d in ("brain", "tests"):
    os.makedirs(os.path.join(W, d), exist_ok=True)
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
DATA = os.path.dirname(glob.glob("/kaggle/input/**/wikitext-103-raw/wiki.train.raw", recursive=True)[0])
print("data:", DATA)

In [ ]:
%%writefile /kaggle/working/brain/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/brain/brain/deepcortex.py
"""A deep language cortex, and three ways it can learn: backprop, predictive coding, and predictive coding with
its own feedback pathway.

Why this exists. The self-trained brain learns in exactly one layer - the striatum choosing among candidates -
and the representations under that choice were written by hand. That is why it cannot understand anything: the
brain's language network is deep, and what makes it deep is that an error at the top reshapes representations
many synapses below. Backprop does that by sending exact gradients backwards along the same weights. Cortex
cannot do that literally (Lillicrap et al. 2020, "Backpropagation and the brain"), and the question this file is
built to answer is how much a *local* rule that the tissue could run gives up against it.

The network is the same for every rule: a neural language model in the style of Bengio et al. (2003) - the last
N words, each as a learned embedding, through L hidden layers, to a prediction of the next word. Predicting the
next word is the cortical computation that best explains the brain's responses to language (Schrimpf et al.
2021; Goldstein et al. 2022; Caucheteux & King 2023), so it is the task.

  bp      exact gradients by autograd. The reference: what the same network learns when credit assignment is
          perfect.

  pc      predictive coding (Rao & Ballard 1999; Whittington & Bogacz 2017; Millidge, Tschantz & Buckley 2022).
          Every layer holds its own activity v_l and predicts the next one up: mu_l = W_l f(v_{l-1}). The
          difference is a prediction-error population, eps_l = v_l - mu_l. When the next word arrives the top is
          wrong, and the activities *settle*: each layer moves to reduce its own error and the error of the layer
          above it (dv_l = -eps_l + f'(v_l) * W_{l+1}^T eps_{l+1}). After settling, every synapse changes by its own
          layer's error times its own input - dW_l = eps_l f(v_{l-1})^T - which is Hebbian between an
          error unit and a presynaptic cell. Nothing is sent backwards through a computation graph; at
          equilibrium the errors approximate the backprop gradients, and the tests check that they do.

  pc-kp   predictive coding, but the feedback that carries eps_{l+1} down to layer l runs through its *own*
          synapses B_{l+1} rather than through W_{l+1}^T. Using the forward weights backwards ("weight
          transport") is the least plausible part of pc: no synapse knows the strength of another synapse
          pointing the other way. The feedback weights learn by the Kolen-Pollack rule (Kolen & Pollack 1994;
          Akrout et al. 2019): the same Hebbian change as the forward synapse, transposed, plus decay on both, so
          the two pathways come into alignment without either reading the other.

Honest notes. The optimiser is Adam for every rule - it is per-synapse (each weight uses only its own history),
so it is local, and using it everywhere keeps the comparison about credit assignment rather than about step
sizes. pc costs `steps` settling iterations per example, so at equal examples it costs roughly that many times
the compute of bp; both budgets are reported.
"""
import math
import time

import torch
import torch.nn.functional as Fn


# ----------------------------------------------------------------------------------------------- text
class Vocab:
    def __init__(self, counts, size):
        words = [w for w, _ in counts.most_common(size - 2)]
        self.itos = ["<unk>", "<eos>"] + [w for w in words if w not in ("<unk>", "<eos>")]
        self.itos = self.itos[:size]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode_lines(self, lines):
        out = []
        for line in lines:
            line = line.strip()
            if not line or line.startswith("="):       # WikiText headings are not sentences
                continue
            out.extend(self.stoi.get(w, 0) for w in line.split())
            out.append(1)
        return torch.tensor(out, dtype=torch.int32)


def read_lines(path, limit_chars=None):
    with open(path, encoding="utf-8") as f:
        text = f.read(limit_chars) if limit_chars else f.read()
    return text.split("\n")


# ----------------------------------------------------------------------------------------------- the network
class Cortex(torch.nn.Module):
    def __init__(self, vocab, context=8, embed=128, hidden=1024, layers=3, seed=0, feedback=False):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.V, self.N, self.d, self.H, self.L = vocab, context, embed, hidden, layers
        self.E = torch.nn.Parameter(torch.randn(vocab, embed, generator=g) * 0.1)
        dims = [context * embed] + [hidden] * layers
        self.W = torch.nn.ParameterList(
            [torch.nn.Parameter(torch.randn(dims[i + 1], dims[i], generator=g) / math.sqrt(dims[i]))
             for i in range(layers)])
        self.b = torch.nn.ParameterList([torch.nn.Parameter(torch.zeros(dims[i + 1])) for i in range(layers)])
        self.Wo = torch.nn.Parameter(torch.randn(vocab, hidden, generator=g) / math.sqrt(hidden))
        self.bo = torch.nn.Parameter(torch.zeros(vocab))
        # the feedback pathway, used only by pc-kp: its own synapses, initialised independently of the forward
        # ones, so any alignment between them has to be learned
        self.feedback = feedback
        if feedback:
            # B[k] carries layer k's error down to whatever feeds it (B[0]: down to the embeddings)
            self.B = torch.nn.ParameterList(
                [torch.nn.Parameter(torch.randn(dims[i], dims[i + 1], generator=g) / math.sqrt(dims[i + 1]))
                 for i in range(layers)])
            self.Bo = torch.nn.Parameter(torch.randn(hidden, vocab, generator=g) / math.sqrt(vocab))

    @staticmethod
    def f(x):
        return torch.tanh(x)

    @staticmethod
    def df(x):
        return 1.0 - torch.tanh(x) ** 2

    def inputs(self, ctx):
        return self.E[ctx.long()].reshape(ctx.shape[0], -1)

    def forward(self, ctx):
        """The feedforward sweep: what every rule uses to predict, and what pc starts settling from."""
        x = self.inputs(ctx)
        v = []
        h = x
        for l in range(self.L):
            a = Fn.linear(h, self.W[l], self.b[l])
            v.append(a)
            h = self.f(a)
        return Fn.linear(h, self.Wo, self.bo), v, x

    def params(self):
        return sum(p.numel() for p in self.parameters())


# ----------------------------------------------------------------------------------------------- learning
def bp_step(net, opt, ctx, y):
    opt.zero_grad(set_to_none=True)
    logits, _, _ = net(ctx)
    loss = Fn.cross_entropy(logits, y.long())
    loss.backward()
    _clip(net)
    opt.step()
    return float(loss.detach())


def _clip(net, limit=1.0):
    """The same gradient-norm limit for every rule, and no step at all on a non-finite one."""
    grads = [p.grad for p in net.parameters() if p.grad is not None]
    total = torch.sqrt(sum((g * g).sum() for g in grads))
    if not torch.isfinite(total):
        for g in grads:
            g.zero_()
        return
    if total > limit:
        for g in grads:
            g.mul_(limit / total)


@torch.no_grad()
def pc_grads(net, ctx, y, steps=20, rate=0.1, beta=0.05):
    """Settle, then compute each synapse's local change. Returns (loss before settling, dict of grads).

    Everything below uses only: a layer's own activity, its own prediction error, the error of the layer
    directly above it (through W^T, or through B in pc-kp), and each synapse's own input. The output error is
    the difference between the predicted distribution and the word that came.

    `beta` is how hard the word that came pulls on the top of the network while it settles. Pulled hard, the
    activities move so far that the errors describe a different network from the one that made the prediction,
    and the updates stop matching the gradient near the top (cosine 0.1-0.3 in the tests). A gentle pull keeps
    the settled state close to the prediction, so the errors are the gradient scaled by beta; dividing by beta
    afterwards restores the size. This is the weak-teaching limit in which predictive coding and equilibrium
    propagation are known to equal backprop (Whittington & Bogacz 2017; Scellier & Bengio 2017).
    """
    logits, v, x = net(ctx)                   # activities start where the feedforward sweep put them
    loss0 = float(Fn.cross_entropy(logits, y.long()))
    y = y.long()
    L = net.L
    v = [a.clone() for a in v]
    n = ctx.shape[0]

    def down(l, err):                         # carry an error from layer l+1 (or the output) down to layer l
        if l == L - 1:
            return err @ (net.Bo.T if net.feedback else net.Wo)
        return err @ (net.B[l + 1].T if net.feedback else net.W[l + 1])

    rows = torch.arange(n, device=x.device)

    def state(v):
        """Everything a settling step needs, and the energy it is trying to lower - per example."""
        mu = [Fn.linear(x, net.W[0], net.b[0])] + [Fn.linear(net.f(v[l - 1]), net.W[l], net.b[l])
                                                  for l in range(1, L)]
        eps = [v[l] - mu[l] for l in range(L)]
        logits = Fn.linear(net.f(v[L - 1]), net.Wo, net.bo)
        p = torch.softmax(logits, dim=1)
        ce = torch.logsumexp(logits, 1) - logits[rows, y]
        p[rows, y] -= 1.0                                        # output error: predicted minus what came
        energy = 0.5 * sum((e * e).sum(1) for e in eps) + beta * ce
        return eps, p, energy

    # Settling must go downhill. The first full run showed why that has to be enforced rather than assumed:
    # as training grew the weights, a step size that had been stable became too large, the activities
    # oscillated off to nowhere, and one batch of garbage errors wrecked the network (pc and pc-kp both rose
    # from ~260-370 to ~2,700 perplexity). A neuron population cannot climb its own energy either. So each
    # example keeps its own step size; a step that would raise its energy is undone and the step halved.
    step = torch.full((n, 1), float(rate), device=x.device)
    eps, p, energy = state(v)
    for _ in range(steps):
        pb = beta * p
        trial = []
        for l in range(L):
            if l == L - 1:
                dv = -eps[l] - net.df(v[l]) * down(l, pb)
            else:
                dv = -eps[l] + net.df(v[l]) * down(l, eps[l + 1])
            trial.append(v[l] + step * dv)
        eps2, p2, energy2 = state(trial)
        ok = (energy2 <= energy) & torch.isfinite(energy2)
        keep = ok[:, None]
        v = [torch.where(keep, t, a) for t, a in zip(trial, v)]
        eps = [torch.where(keep, e2, e) for e2, e in zip(eps2, eps)]
        p = torch.where(keep, p2, p)
        energy = torch.where(ok, energy2, energy)
        step = torch.where(keep, step, step * 0.5)
    h_top = net.f(v[L - 1])
    eps = [e / beta for e in eps]             # the errors were raised by a pull of size beta
    grads = {}
    # gradient-shaped quantities (the negative of each synapse's Hebbian change), averaged over the batch
    grads["Wo"] = p.T @ h_top / n
    grads["bo"] = p.mean(0)
    for l in range(L):
        pre = x if l == 0 else net.f(v[l - 1])
        grads[f"W{l}"] = -(eps[l].T @ pre) / n
        grads[f"b{l}"] = -eps[l].mean(0)
    # the embeddings are the synapses from the word's identity onto the input layer; their error arrives from
    # layer 1 the same way any other layer's does
    back = eps[0] @ (net.B[0].T if net.feedback else net.W[0])
    grads["E"] = (-back / n, ctx)
    if net.feedback:
        # Kolen-Pollack: each feedback synapse gets the forward synapse's change, transposed
        for l in range(L):
            grads[f"B{l}"] = grads[f"W{l}"].T
        grads["Bo"] = grads["Wo"].T
    return loss0, grads


def pc_step(net, opt, ctx, y, steps=20, rate=0.1, beta=0.05):
    loss0, g = pc_grads(net, ctx, y, steps, rate, beta)
    opt.zero_grad(set_to_none=True)
    net.Wo.grad, net.bo.grad = g["Wo"], g["bo"]
    for l in range(net.L):
        net.W[l].grad, net.b[l].grad = g[f"W{l}"], g[f"b{l}"]
    back, c = g["E"]
    gE = torch.zeros_like(net.E)
    gE.index_add_(0, c.long().reshape(-1), back.reshape(-1, net.d))
    net.E.grad = gE
    if net.feedback:
        for l in range(net.L):
            net.B[l].grad = g[f"B{l}"]
        net.Bo.grad = g["Bo"]
    _clip(net)
    opt.step()
    return loss0


@torch.no_grad()
def shrink(net, factor):
    """Decoupled weight decay, applied the same way to every rule.

    For pc-kp this is what does the aligning (Kolen-Pollack): a feedback synapse receives exactly the forward
    synapse's change, transposed, so Adam moves the two identically, and multiplying both by the same factor
    every step makes whatever difference they were born with fade away. Added to the gradient instead, Adam
    normalises it out and the pathways never meet - which is what the first pilot did (alignment 0.08-0.34).
    """
    mats = [*net.W, net.Wo] + ([*net.B, net.Bo] if net.feedback else [])
    for m in mats:
        m.mul_(factor)


# ----------------------------------------------------------------------------------------------- measuring
@torch.no_grad()
def perplexity(net, tokens, batch=2048, limit=None):
    N = net.N
    idx = torch.arange(N, len(tokens) if not limit else min(len(tokens), N + limit), device=tokens.device)
    total, count = 0.0, 0
    for s in range(0, len(idx), batch):
        i = idx[s:s + batch]
        ctx = tokens[i[:, None] - N + torch.arange(N, device=tokens.device)]
        logits, _, _ = net(ctx)
        total += float(Fn.cross_entropy(logits, tokens[i].long(), reduction="sum"))
        count += len(i)
    return math.exp(total / max(1, count))


def batch_of(tokens, n, N, gen):
    i = torch.randint(N, len(tokens), (n,), generator=gen, device="cpu").to(tokens.device)
    ctx = tokens[i[:, None] - N + torch.arange(N, device=tokens.device)]
    return ctx, tokens[i]


@torch.no_grad()
def alignment(net):
    """How well each learned feedback pathway has come to match the forward one (cosine), for pc-kp."""
    if not net.feedback:
        return None
    out = [float(Fn.cosine_similarity(net.B[l].flatten(), net.W[l].T.flatten(), dim=0)) for l in range(net.L)]
    out.append(float(Fn.cosine_similarity(net.Bo.flatten(), net.Wo.T.flatten(), dim=0)))
    return out


def train(rule, train_tokens, valid_tokens, vocab, steps, batch=512, lr=1e-3, seed=0, context=8, embed=128,
          hidden=1024, layers=3, pc_steps=20, pc_rate=0.2, pc_beta=0.05, weight_decay=0.1, eval_every=1000,
          valid_limit=100000,
          device="cpu", log=print, deadline=None):
    torch.manual_seed(seed)
    net = Cortex(vocab, context, embed, hidden, layers, seed=seed, feedback=(rule == "pc-kp")).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(1000 + seed)
    curve = []
    t0 = time.time()
    run = 0.0
    for step in range(1, steps + 1):
        ctx, y = batch_of(train_tokens, batch, context, gen)
        if rule == "bp":
            loss = bp_step(net, opt, ctx, y)
        else:
            loss = pc_step(net, opt, ctx, y, pc_steps, pc_rate, beta=pc_beta)
        if weight_decay:
            shrink(net, 1.0 - lr * weight_decay)
        run = loss if step == 1 else 0.98 * run + 0.02 * loss
        if step % eval_every == 0 or step == steps:
            ppl = perplexity(net, valid_tokens, limit=valid_limit)
            entry = {"step": step, "examples": step * batch, "train_loss": round(run, 4), "valid_ppl": round(ppl, 2),
                     "seconds": round(time.time() - t0, 1)}
            if rule == "pc-kp":
                entry["alignment"] = [round(a, 3) for a in alignment(net)]
            curve.append(entry)
            log(f"  [{rule} seed {seed} lr {lr}] {entry}")
            if not math.isfinite(ppl) or ppl > 1e5:
                log("  diverged - stopping this run")
                break
        if deadline and time.time() > deadline:
            log("  out of time - stopping this run where it is")
            break
    return net, curve


@torch.no_grad()
def generate(net, vocab, prompt, words=30, temperature=0.8, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = [vocab.stoi.get(w, 0) for w in prompt.split()]
    ids = [1] * max(0, net.N - len(ids)) + ids
    out = []
    dev = next(net.parameters()).device
    for _ in range(words):
        ctx = torch.tensor([ids[-net.N:]], device=dev)
        logits, _, _ = net(ctx)
        p = torch.softmax(logits[0] / temperature, 0).cpu()
        p[0] = 0                                      # never say <unk>
        w = int(torch.multinomial(p / p.sum(), 1, generator=g))
        ids.append(w)
        out.append(vocab.itos[w])
    return " ".join(out).replace(" <eos> ", "\n")

In [ ]:
%%writefile /kaggle/working/brain/brain/kneser.py
"""The counting baseline: interpolated Kneser-Ney (Kneser & Ney 1995; Chen & Goodman 1999).

Any claim that a learned cortex predicts the next word well has to beat counting, because counting is what the
brain's earlier self-trained parts did and what an n-gram model has done since the 1990s. KN is the strongest
classical form of it: absolute discounting, and lower orders estimated from how many *different* contexts a word
follows ("Francisco" is common but almost only after "San", so it should be a poor guess after anything else).

Vectorised: every n-gram is folded into one 64-bit key (a random polynomial hash; collisions are negligible at
these sizes), counted with numpy.unique, and looked up with searchsorted, so tens of millions of tokens fit in a
few minutes and a few GB.
"""
import math

import numpy as np

_M = np.random.default_rng(12345).integers(1, 2 ** 63 - 1, size=16, dtype=np.int64).astype(np.uint64) | np.uint64(1)


def _keys(tokens, k, end):
    """Key of the k tokens ending at each position in `end` (exclusive)."""
    key = np.zeros(len(end), np.uint64)
    t = tokens.astype(np.uint64)
    for j in range(k):
        key = key * np.uint64(1000003) + (t[end - k + j] + np.uint64(1)) * _M[j]
    return key + np.uint64(k)


class Table:
    def __init__(self, keys, values):
        order = np.argsort(keys)
        self.k, self.v = keys[order], values[order]

    def get(self, q):
        i = np.searchsorted(self.k, q)
        i = np.minimum(i, len(self.k) - 1)
        hit = self.k[i] == q
        out = np.zeros(len(q), self.v.dtype if self.v.ndim == 1 else self.v.dtype)
        if self.v.ndim == 1:
            out = np.where(hit, self.v[i], 0)
        else:
            out = np.where(hit[:, None], self.v[i], 0)
        return out


class KneserNey:
    def __init__(self, order=5):
        self.n = order

    def fit(self, tokens, vocab):
        tokens = np.asarray(tokens, np.int64)
        self.V = vocab
        T = len(tokens)
        self.ngram, self.context, self.D = {}, {}, {}
        for k in range(1, self.n + 1):
            end = np.arange(k, T + 1)
            g = _keys(tokens, k, end)
            if k == self.n:
                # the highest order uses raw counts
                uniq, first, counts = np.unique(g, return_index=True, return_counts=True)
                ends = end[first]
            else:
                # lower orders use continuation counts: how many distinct words came before this k-gram
                end1 = np.arange(k + 1, T + 1)
                g1 = _keys(tokens, k + 1, end1)
                u1, f1 = np.unique(g1, return_index=True)
                suffix = _keys(tokens, k, end1[f1])
                uniq, first, counts = np.unique(suffix, return_index=True, return_counts=True)
                ends = end1[f1][first]
            n1, n2 = int((counts == 1).sum()), int((counts == 2).sum())
            self.D[k] = n1 / (n1 + 2 * n2) if n1 + 2 * n2 else 0.5
            self.ngram[k] = Table(uniq, counts.astype(np.float64))
            if k == 1:
                self.unigram_total = float(counts.sum())
                self.unigram_types = float(len(counts))
            else:
                ctx = _keys(tokens, k - 1, ends - 1)
                cu, inv = np.unique(ctx, return_inverse=True)
                total = np.bincount(inv, weights=counts.astype(np.float64))
                types = np.bincount(inv).astype(np.float64)
                self.context[k] = Table(cu, np.stack([total, types], 1))
        return self

    def logprob(self, tokens, start):
        """log P(tokens[i] | the n-1 before it) for every i >= start."""
        tokens = np.asarray(tokens, np.int64)
        pos = np.arange(start, len(tokens))
        D1 = self.D[1]
        c1 = self.ngram[1].get(_keys(tokens, 1, pos + 1))
        p = (np.maximum(c1 - D1, 0) + D1 * self.unigram_types / self.V) / self.unigram_total
        for k in range(2, self.n + 1):
            ok = pos - (k - 1) >= 0
            c = self.ngram[k].get(_keys(tokens, k, pos + 1))
            ctx = self.context[k].get(_keys(tokens, k - 1, pos))
            total, types = ctx[:, 0], ctx[:, 1]
            seen = (total > 0) & ok
            D = self.D[k]
            higher = np.where(seen, (np.maximum(c - D, 0) + D * types * p) / np.where(seen, total, 1), p)
            p = higher
        return np.log(np.maximum(p, 1e-12))

    def perplexity(self, tokens, start=8):
        return math.exp(-float(self.logprob(tokens, start).mean()))

In [ ]:
%%writefile /kaggle/working/brain/brain/learn_cortex.py
"""Train the deep language cortex three ways and measure what brain-plausible credit assignment costs.

    python brain/learn_cortex.py --data <dir with wiki.train.raw/valid/test> --steps 20000 --seeds 5 --out results/cortex

Pre-registered in PREREGISTRATION_deepcortex.md. In order:

  1. vocabulary (most frequent words of the training text) and tokens
  2. the counting baseline: Kneser-Ney 5-gram on the same number of training tokens the networks will see
  3. learning-rate selection: every rule gets the same small grid on a short run, chosen on validation - the
     baseline is tuned exactly as hard as the proposal
  4. the main runs, seeds interleaved across rules (seed 0 for all three, then seed 1 ...) so that if the session
     runs out of time the seeds that finished are balanced
  5. held-out test perplexity for every finished run, the paired tests, samples, and the best network of each
     rule saved so the brain can use it as its language cortex
"""
import argparse
import collections
import json
import math
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))

import numpy as np  # noqa: E402
import torch  # noqa: E402

from brain import deepcortex as D  # noqa: E402
from brain.kneser import KneserNey  # noqa: E402

RULES = ("bp", "pc", "pc-kp")


def load(data, vocab_size, log):
    tr = D.read_lines(os.path.join(data, "wiki.train.raw"))
    va = D.read_lines(os.path.join(data, "wiki.valid.raw"))
    te = D.read_lines(os.path.join(data, "wiki.test.raw"))
    counts = collections.Counter()
    for line in tr:
        counts.update(line.split())
    voc = D.Vocab(counts, vocab_size)
    T, V, X = voc.encode_lines(tr), voc.encode_lines(va), voc.encode_lines(te)
    log(f"vocabulary {len(voc):,}; tokens train {len(T):,} valid {len(V):,} test {len(X):,}; "
        f"unknown in test {float((X == 0).float().mean()):.3%}")
    return voc, T, V, X


def analyse(runs, kn_test, alpha=0.05):
    from scipy import stats
    by = collections.defaultdict(dict)
    for r in runs:
        if r.get("test_ppl"):
            by[r["rule"]][r["seed"]] = r["test_ppl"]
    both = sorted(set(by["bp"]) & set(by["pc"]))
    tests = []
    for rule in ("pc", "pc-kp"):
        x = np.log([by[rule][s] for s in sorted(by[rule])])
        if len(x) >= 2:
            t, p = stats.ttest_1samp(x, math.log(kn_test))
            tests.append((f"H{1 if rule == 'pc' else 2} {rule} beats Kneser-Ney 5-gram (test ppl)",
                          float(math.log(kn_test) - x.mean()), float(-t), float(p / 2 if t < 0 else 1 - p / 2)))
    if len(both) >= 2:
        gap = np.log([by["pc"][s] for s in both]) - np.log([by["bp"][s] for s in both])
        t, p = stats.ttest_1samp(gap, math.log(1.25))
        tests.append(("H3 pc within 25% of bp (test ppl ratio < 1.25)", float(math.log(1.25) - gap.mean()),
                      float(-t), float(p / 2 if t < 0 else 1 - p / 2)))
    order = sorted(range(len(tests)), key=lambda i: tests[i][3])
    verdict, alive = {}, True
    for rank, i in enumerate(order):
        name, diff, t, p = tests[i]
        alive = alive and p <= alpha / (len(tests) - rank) and diff > 0
        verdict[name] = "SUPPORTED" if alive else "not supported"
    means = {rule: {"test_ppl": round(float(np.exp(np.mean(np.log(list(v.values()))))), 2), "seeds": len(v)}
             for rule, v in by.items()}
    return {"tests": [dict(zip(("name", "margin_log", "t", "p_one_sided"), x)) for x in tests],
            "verdict": verdict, "geometric_mean_test_ppl": means, "kneser_ney_test_ppl": kn_test}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/cortex")
    ap.add_argument("--vocab", type=int, default=32000)
    ap.add_argument("--steps", type=int, default=20000)
    ap.add_argument("--batch", type=int, default=512)
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--hidden", type=int, default=1024)
    ap.add_argument("--embed", type=int, default=128)
    ap.add_argument("--layers", type=int, default=3)
    ap.add_argument("--context", type=int, default=8)
    ap.add_argument("--pc-steps", type=int, default=40)
    ap.add_argument("--pc-rate", type=float, default=0.2)
    ap.add_argument("--pc-beta", type=float, default=0.01)
    ap.add_argument("--weight-decay", type=float, default=0.1)
    ap.add_argument("--grid", type=float, nargs="*", default=[3e-4, 1e-3, 3e-3])
    ap.add_argument("--grid-steps", type=int, default=2000)
    ap.add_argument("--hours", type=float, default=10.0)
    ap.add_argument("--rules", nargs="*", default=list(RULES))
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    deadline = time.time() + a.hours * 3600
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    logf = open(os.path.join(a.out, "log.txt"), "a", encoding="utf-8")

    def log(*x):
        line = " ".join(str(v) for v in x)
        print(line, flush=True)
        logf.write(line + "\n")
        logf.flush()

    log(f"device {dev}" + (f" ({torch.cuda.get_device_name(0)})" if dev == "cuda" else ""))
    voc, T, V, X = load(a.data, a.vocab, log)
    budget = a.steps * a.batch

    # 2. counting, on as many training tokens as the networks will see examples
    t0 = time.time()
    kn = KneserNey(5).fit(T[:budget].numpy(), len(voc))
    kn_valid, kn_test = kn.perplexity(V.numpy()), kn.perplexity(X.numpy())
    log(f"Kneser-Ney 5-gram on {min(budget, len(T)):,} tokens: valid {kn_valid:.2f} test {kn_test:.2f} "
        f"({time.time() - t0:.0f}s)")
    del kn
    T, V, X = T.to(dev), V.to(dev), X.to(dev)
    common = dict(batch=a.batch, context=a.context, embed=a.embed, hidden=a.hidden, layers=a.layers,
                  pc_steps=a.pc_steps, pc_rate=a.pc_rate, pc_beta=a.pc_beta, weight_decay=a.weight_decay,
                  device=dev, log=log)
    results = {"config": vars(a), "vocab": len(voc), "kn": {"valid": kn_valid, "test": kn_test}, "grid": [],
               "runs": []}

    def save():
        json.dump(results, open(os.path.join(a.out, "results.json"), "w"), indent=1)

    # 3. the same learning-rate grid for every rule, chosen on validation
    chosen = {}
    for rule in a.rules:
        best = None
        for lr in a.grid:
            net, curve = D.train(rule, T, V, len(voc), steps=a.grid_steps, lr=lr, seed=0,
                                 eval_every=a.grid_steps, valid_limit=100000, **common)
            ppl = curve[-1]["valid_ppl"] if curve else float("inf")
            results["grid"].append({"rule": rule, "lr": lr, "valid_ppl": ppl, "seconds": curve[-1]["seconds"]
                                    if curve else None})
            if best is None or ppl < best[1]:
                best = (lr, ppl)
            del net
        chosen[rule] = best[0]
        log(f"learning rate for {rule}: {best[0]} (valid {best[1]})")
        save()
    results["chosen_lr"] = chosen

    # 4. the main runs, seeds interleaved
    best_net = {}
    for seed in range(a.seeds):
        for rule in a.rules:
            left = deadline - time.time()
            if left < 600:
                log(f"stopping: {left / 60:.0f} min left")
                break
            t0 = time.time()
            net, curve = D.train(rule, T, V, len(voc), steps=a.steps, lr=chosen[rule], seed=seed,
                                 eval_every=max(1000, a.steps // 10), valid_limit=None, deadline=deadline - 300,
                                 **common)
            finished = bool(curve) and curve[-1]["step"] == a.steps
            test = D.perplexity(net, X) if finished else None
            run = {"rule": rule, "seed": seed, "lr": chosen[rule], "finished": finished, "curve": curve,
                   "test_ppl": round(test, 2) if test else None, "seconds": round(time.time() - t0),
                   "params": net.params(), "samples": [D.generate(net, voc, p, words=25, seed=seed)
                                                       for p in ("The city is", "In 1918 , the", "He was born in")]}
            if rule == "pc-kp":
                run["alignment"] = D.alignment(net)
            results["runs"].append(run)
            log(f"== {rule} seed {seed}: test ppl {run['test_ppl']} ({run['seconds']}s) | {run['samples'][0]}")
            if finished and (rule not in best_net or test < best_net[rule][0]):
                best_net[rule] = (test, seed)
                torch.save({"state": {k: v.cpu() for k, v in net.state_dict().items()}, "itos": voc.itos,
                            "config": {"context": a.context, "embed": a.embed, "hidden": a.hidden,
                                       "layers": a.layers, "feedback": rule == "pc-kp"},
                            "rule": rule, "test_ppl": test},
                           os.path.join(a.out, f"cortex_{rule}.pt"))
            del net
            save()
        else:
            continue
        break

    # 5. the pre-registered tests
    results["analysis"] = analyse(results["runs"], kn_test)
    save()
    log(json.dumps(results["analysis"], indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/brain/tests/check_deepcortex.py
"""The deep cortex's learning rules, checked where they can be checked exactly.

The claim of predictive coding is that local changes, after settling, point where backprop's gradients point.
That is testable on a small network in a second, and if it failed nothing trained with it would mean anything.
"""
import os
import sys

import numpy as np
import torch
import torch.nn.functional as Fn

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))

from brain.deepcortex import Cortex, pc_grads, shrink, perplexity  # noqa: E402
from brain.kneser import KneserNey  # noqa: E402

FAILED = []


def check(name, cond, detail=""):
    print(f"{'PASS' if cond else 'FAIL'}  {name}" + (f"   {detail}" if detail and not cond else ""))
    if not cond:
        FAILED.append(name)


def cos(a, b):
    return float(Fn.cosine_similarity(a.flatten(), b.flatten(), dim=0))


torch.manual_seed(0)
net = Cortex(50, context=3, embed=8, hidden=32, layers=3, seed=1)
ctx = torch.randint(0, 50, (64, 3))
y = torch.randint(0, 50, (64,))
logits, _, _ = net(ctx)
Fn.cross_entropy(logits, y).backward()
_, g = pc_grads(net, ctx, y, steps=20, rate=0.2, beta=0.05)
check("pc output synapses point where backprop's gradient points", cos(g["Wo"], net.Wo.grad) > 0.95,
      f"{cos(g['Wo'], net.Wo.grad):.3f}")
for l in range(3):
    c = cos(g[f"W{l}"], net.W[l].grad)
    check(f"pc layer {l} synapses point where backprop's gradient points", c > 0.9, f"{c:.3f}")
gE = torch.zeros_like(net.E)
back, c = g["E"]
gE.index_add_(0, c.reshape(-1), back.reshape(-1, net.d))
check("pc embedding synapses point where backprop's gradient points", cos(gE, net.E.grad) > 0.9,
      f"{cos(gE, net.E.grad):.3f}")
_, strong = pc_grads(net, ctx, y, steps=20, rate=0.2, beta=1.0)
check("and a hard pull at the top is what breaks it (the reason for beta)", cos(strong["Wo"], net.Wo.grad) < 0.9,
      f"{cos(strong['Wo'], net.Wo.grad):.3f}")

# the failure of the first full run, reproduced on purpose: weights grown large and a step size far too big.
# Without the downhill rule the activities run off to infinity; with it the errors stay finite and still point
# roughly where backprop's gradient does.
big = Cortex(50, context=3, embed=8, hidden=32, layers=3, seed=3)
with torch.no_grad():
    for W_ in big.W:
        W_.mul_(4.0)
logits, _, _ = big(ctx)
Fn.cross_entropy(logits, y).backward()
_, gb = pc_grads(big, ctx, y, steps=40, rate=1.5, beta=0.01)
finite = all(torch.isfinite(v if not isinstance(v, tuple) else v[0]).all() for v in gb.values())
check("with grown weights and a reckless step size, settling stays finite", bool(finite))
check("and its updates still point where backprop's do", cos(gb["Wo"], big.Wo.grad) > 0.8,
      f"{cos(gb['Wo'], big.Wo.grad):.3f}")

# pc-kp: feedback synapses receive the forward change transposed, and shrinking both aligns them
kp = Cortex(50, context=3, embed=8, hidden=32, layers=2, seed=2, feedback=True)
before = cos(kp.B[1], kp.W[1].T)
_, gk = pc_grads(kp, ctx, y, steps=20, rate=0.2, beta=0.05)
check("a feedback synapse is told the forward synapse's change, transposed", torch.allclose(gk["B1"], gk["W1"].T))
opt = torch.optim.Adam(kp.parameters(), lr=1e-2)
from brain.deepcortex import pc_step  # noqa: E402
for _ in range(300):
    pc_step(kp, opt, ctx, y, steps=10, rate=0.2, beta=0.05)
    shrink(kp, 1.0 - 1e-2 * 1.0)
after = cos(kp.B[1], kp.W[1].T)
check("and the two pathways come into alignment without reading each other", after > before + 0.3,
      f"{before:.3f} -> {after:.3f}")

# the counting baseline is a probability distribution
rng = np.random.default_rng(0)
toks = rng.integers(0, 30, size=5000)
kn = KneserNey(4).fit(toks, 30)
seqs = np.concatenate([np.tile(toks[100:103], (30, 1)), np.arange(30)[:, None]], 1)
total = sum(float(np.exp(kn.logprob(s, 3))[0]) for s in seqs)
check("Kneser-Ney sums to one over the vocabulary", abs(total - 1) < 1e-6, f"{total}")
check("perplexity is finite and above one", 1 < perplexity(net, torch.randint(0, 50, (500,)).int()) < 1e4)

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/brain/PREREGISTRATION_deepcortex.md
# Pre-registration: what does brain-plausible credit assignment cost a deep language cortex?

Written 2026-09-23, before the Kaggle run of `brain/learn_cortex.py`. Code at the commit this file is in.

## Why

The self-trained brain learns in one layer (the striatum) over hand-written features, and it does not understand
language. The brain's language network is deep, and learning in a deep network needs credit assignment through
many layers. Backprop does that exactly but is not something cortex can literally do (Lillicrap et al. 2020).
Predictive coding (PC) does it with local rules and approximates backprop (Whittington & Bogacz 2017; Millidge et
al. 2022); PC with learned feedback (Kolen-Pollack, "pc-kp") removes the weight-transport problem as well. The
question is what each step toward plausibility costs on real language, measured against counting.

## The network and the rules

One network for every rule: a neural language model (Bengio et al. 2003) - the previous 8 words as 128-d
embeddings, 3 tanh layers of 1024, softmax over a 32,000-word vocabulary built from WikiText-103 train (all
other words `<unk>`). Rules:

- **bp**: autograd.
- **pc**: activities settle for 40 steps (rate 0.2) under a weak teaching signal (beta 0.01), then each synapse
  changes by its own layer's prediction error times its own input; errors are rescaled by 1/beta.
- **pc-kp**: pc, with every feedback projection its own matrix, receiving the forward synapse's change
  transposed; decoupled weight decay aligns the two.

All rules: Adam, decoupled weight decay 0.1 on all weight matrices, batch 512, 20,000 steps (10.24M examples,
sampled uniformly from the training tokens). Learning rate per rule from {3e-4, 1e-3, 3e-3} on a 2,000-step run
with seed 0, chosen on validation perplexity - the same procedure for every rule.

Baseline: interpolated Kneser-Ney 5-gram (discounts n1/(n1+2n2) per order) on the first 10.24M training tokens -
as many tokens as the networks see examples. Same vocabulary and `<unk>` handling.

## Checks already passing (tests/check_deepcortex.py)

PC's settled updates have cosine > 0.95 with backprop's gradients at every layer of a small network at beta 0.05,
and < 0.9 at the output with beta 1 (why beta is needed). A pc-kp feedback synapse receives exactly the forward
change transposed, and alignment rises with training. Kneser-Ney sums to one over the vocabulary.

## Exploratory pilots (local CPU, WikiText-2, 10k vocabulary, 256 hidden, 1,500 steps x 128; one seed)

Validation perplexity: bp 220, pc 269, pc-kp 545 with weight decay added to the gradient (feedback alignment
0.08-0.34, i.e. it never aligned). Changed before this file: decay made decoupled (see `deepcortex.shrink`).
Kneser-Ney 5-gram on the same WikiText-2 train: 117.7 - counting wins at that data size.
Second pilot, same settings: bp 220.5; pc with 40 settling steps and beta 0.01 235.7 (chosen for the run;
20 steps / beta 0.05 gave 269); pc-kp with decoupled decay 1.0: 303 and still falling, alignment 0.51-0.96;
decay 3.0: aligned (0.99) but 507. The run uses decay 0.1 for every rule over 20,000 steps, which shrinks a
misalignment by about the same total factor as 1.0 over the pilot's 1,500.

Compute: bp and pc run on one GPU, pc-kp on a second, so both PC arms fit the session.

## Hypotheses (test perplexity on WikiText-103 test, finished runs, up to 5 seeds)

- **H1.** pc beats Kneser-Ney: mean log test perplexity of pc below log KN.
- **H2.** pc-kp beats Kneser-Ney.
- **H3.** pc is within 25% of bp: mean of log(ppl_pc / ppl_bp) over paired seeds below log 1.25.

One-sided t-tests on log perplexity (H1, H2 one-sample against the deterministic KN value; H3 one-sample on paired
log-ratios against log 1.25), Holm-Bonferroni across the three, alpha 0.05.

Reported regardless: bp vs KN, pc-kp vs pc, compute per rule (seconds), feedback alignment for pc-kp, samples.

## What would count as failure

H1 failing means a brain-plausible deep cortex does not beat counting at this budget, and there is no case yet
for making it the brain's language cortex. H3 failing means plausibility costs more than a quarter of the
perplexity. If only bp beats KN, the cortex the brain uses will be bp-trained, and that is said plainly.

## Amendment 1, 2026-09-24, after the first run and before the second

**First run (reported in full, not used for the hypotheses).** WikiText-103, 32k vocabulary, 3 x 1024, 20,000
steps. Kneser-Ney 5-gram test perplexity 177.7. bp: 166.4 and 165.5 test (2 seeds). pc seed 1 tracked bp for
4,000 steps (valid 336 then 261, against bp's 321 then 255) and then diverged to ~14,700 and settled near
2,900; pc seed 0 diverged before step 2,000; pc-kp reached 371 with feedback aligned 0.90-0.99 and then
diverged at step 16,000. pc cost 16,000 s per run (18x bp), so only 1-2 seeds finished and H1-H3 were not
computed.

**Diagnosis.** The settling step size was fixed; as the weights grew, it exceeded what the dynamics could take,
the activities oscillated off, and a single batch of garbage errors wrecked the network.

**Changes, applied to every rule alike.**
- Settling must lower the energy: each example keeps its own step size, a step that would raise the example's
  energy (0.5 * sum of squared prediction errors + beta * cross-entropy) is undone and its step halved. A test
  reproduces the failure (weights x4, step 1.5) and checks settling stays finite and aligned with backprop.
- Every rule's update is clipped to global norm 1.0, and a non-finite update is skipped.
- To fit five seeds of both PC arms in one session: 3 hidden layers of 512 (was 1024) and a 16,000-word
  vocabulary (was 32,000). Everything else, the grid, the hypotheses and the analysis, is unchanged.

### Checks

In [ ]:
r = subprocess.run([sys.executable, "tests/check_deepcortex.py"], capture_output=True, text=True, cwd=W)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "checks failed"

### The experiment

In [ ]:
# two GPUs, two processes: bp and pc on one, pc-kp on the other; each chooses its own learning rates
# by the same grid and both compute the same Kneser-Ney baseline
import torch
gpus = torch.cuda.device_count()
print("GPUs:", gpus)
jobs = [("0", ["bp", "pc"], "/kaggle/working/cortex_a"), ("1" if gpus > 1 else "0", ["pc-kp"], "/kaggle/working/cortex_b")]
procs = []
for dev, rules, out in jobs:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=dev)
    cmd = [sys.executable, "-u", "brain/learn_cortex.py", "--data", DATA, "--out", out, "--rules", *rules,
           "--steps", "20000", "--seeds", "5", "--hours", "11.0",
           "--hidden", "512", "--layers", "3", "--vocab", "16000"]
    procs.append(subprocess.Popen(cmd, cwd=W, env=env, stdout=open(out + ".log", "w"), stderr=subprocess.STDOUT))
start = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    for _, _, out in jobs:
        lines = open(out + ".log").read().splitlines()
        print(f"[{(time.time() - start) / 3600:.1f}h] {out[-8:]}:", *[l for l in lines if l.startswith(("==", "learning rate", "Kneser", "vocab", "device"))][-3:], sep="\n  ", flush=True)
for _, _, out in jobs:
    print(open(out + ".log").read()[-3000:])

### Curves and verdicts

In [ ]:
import json, sys
sys.path.insert(0, W)
from brain.learn_cortex import analyse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
A = json.load(open("/kaggle/working/cortex_a/results.json"))
B = json.load(open("/kaggle/working/cortex_b/results.json"))
runs = A["runs"] + B["runs"]
analysis = analyse(runs, A["kn"]["test"])
json.dump({"kn": A["kn"], "grid": A["grid"] + B["grid"], "chosen_lr": {**A.get("chosen_lr", {}), **B.get("chosen_lr", {})},
           "runs": runs, "analysis": analysis}, open("/kaggle/working/cortex_results.json", "w"), indent=1)
fig, ax = plt.subplots(figsize=(7, 4))
colors = {"bp": "C0", "pc": "C1", "pc-kp": "C2"}
for r in runs:
    ax.plot([c["examples"] / 1e6 for c in r["curve"]], [c["valid_ppl"] for c in r["curve"]], color=colors[r["rule"]],
            alpha=0.6, label=r["rule"] if r["seed"] == 0 else None)
ax.axhline(A["kn"]["valid"], ls="--", c="gray", label="Kneser-Ney 5-gram")
ax.set_yscale("log"); ax.set_xlabel("training examples (millions)"); ax.set_ylabel("validation perplexity")
ax.legend(); plt.tight_layout(); plt.savefig("/kaggle/working/cortex_curves.png", dpi=120)
print(json.dumps(analysis, indent=1))
for r in runs:
    print(r["rule"], r["seed"], r["test_ppl"], r.get("alignment"), "|", r["samples"][0])